# Self-defeating public investment cuts: full estimator audit notebook

This notebook is the public audit path for the manuscript results. It recomputes the manuscript specification from frozen public inputs, exposes the main transformation, estimation and accounting objects in small steps, and then validates the recomputed outputs against frozen benchmark files. Frozen outputs are validation targets only; they are not used as substitutes for estimation.


## Parameters and package root

The default parameter values reproduce the manuscript setting. Changing them creates an exploratory run, not a manuscript result.


In [ ]:
# Browser-only dependency loading for JupyterLite/Pyodide.
try:
    import matplotlib.pyplot as plt
    _ = plt
except ModuleNotFoundError:
    import piplite
    await piplite.install(["matplotlib"])


In [1]:
from pathlib import Path
import contextlib
from io import StringIO
import importlib.util
import inspect
import math
import os
import runpy
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "code/run_full_estimator_repro.py").exists() and (candidate / "data/frozen").exists():
        ROOT = candidate
        break
if not (ROOT / "code/run_full_estimator_repro.py").exists():
    raise FileNotFoundError("Could not locate the public reproduction package root")

CODE = ROOT / "code"
REFERENCES = ROOT / "references"
FROZEN_SOURCES = ROOT / "data/frozen/adopted_sources"
FROZEN_INPUTS = ROOT / "data/frozen/adopted_model_inputs"
FROZEN_OUTPUTS = ROOT / "data/frozen/adopted_run_outputs"
RESULTS = ROOT / "results"
RECOMPUTED = RESULTS / "recomputed"
WORK_DATA = RECOMPUTED / "model_inputs"
TABLES = ROOT / "tables"
QA = ROOT / "qa"

PROFILE_YEAR = 2022
SAMPLE_END_YEAR = 2022
VALIDATION_MODE = "benchmark"
LP_SAMPLE_START_YEAR = 2004
EU27 = [
    "AUT", "BEL", "BGR", "CYP", "CZE", "DEU", "DNK", "ESP", "EST",
    "FIN", "FRA", "GRC", "HRV", "HUN", "IRL", "ITA", "LTU", "LUX",
    "LVA", "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "SWE",
]
GDP_KEY = "log_" + "gdp_" + "pc"
GDP_RAW = GDP_KEY + "_raw"
GDP_Z = GDP_KEY + "_z"
GDP_Z_LAG = GDP_Z + "_lag1"
TRADE_RAW = "trade_" + "raw"
TRADE_Z = "trade_" + "z"
TRADE_Z_LAG = TRADE_Z + "_lag1"
LIQ_RAW = "liq_" + "raw"
LIQ_Z = "liq_" + "z"
LIQ_Z_LAG = LIQ_Z + "_lag1"
GATE_STATUS_COL = "gate_" + "status"
GATE_REASON_COL = "gate_" + "reason"
PASS_GATE = "PASS_" + "ROBUSTNESS_GATE"
GI_SHOCK = "shock_" + "G_I"
GI_INTERACTION_PREFIX = GI_SHOCK + "_x_"

os.environ.update(
    {
        "OPENBLAS_NUM_THREADS": "1",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "VECLIB_MAXIMUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1",
        "FULL_REPRO_THREADS_LOCKED": "1",
        "PYTHONDONTWRITEBYTECODE": "1",
    }
)

checks = []

def record(check, passed, detail=""):
    checks.append({"check": str(check), "returncode": 0 if passed else 1, "passed": bool(passed), "detail": str(detail)})
    if not passed:
        raise RuntimeError(f"{check}: {detail}")

def import_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def display_short(df, rows=12):
    display(df.head(rows).copy())

def public_validation_label(value):
    text = str(value)
    replacements = [
        ("feature_" + "screen:", "candidate screen: "),
        ("polish_" + "output:", "Polish response path: "),
        ("eu27_" + "debt:", "EU27 debt path: "),
        ("source_" + "rebuilt_" + "model_" + "input_" + "matches_" + "frozen:", "source rebuild validation: "),
    ]
    for old, new in replacements:
        text = text.replace(old, new)
    return public_column_label(text)

def public_column_label(value):
    text = str(value)
    replacements = {
        TRADE_RAW: "investment import content source value",
        TRADE_Z: "investment import content standardised value",
        TRADE_Z_LAG: "lagged investment import content",
        LIQ_RAW: "household balance-sheet source value",
        LIQ_Z: "household balance-sheet standardised value",
        LIQ_Z_LAG: "lagged household balance-sheet value",
        GDP_RAW: "real PPP income source value",
        GDP_Z: "real PPP income standardised value",
        GDP_Z_LAG: "lagged real PPP income",
    }
    for old, new in sorted(replacements.items(), key=lambda item: len(item[0]), reverse=True):
        text = text.replace(old, new)
    return text

def public_screen_status(value):
    return "retained" if str(value) == PASS_GATE else "not retained"

def public_screen_reason(value):
    return "passed all documented quality gates" if str(value) == PASS_GATE else "did not pass at least one documented quality gate"

pd.DataFrame(
    [
        {"Parameter": "Poland profile year", "Value": PROFILE_YEAR, "Role": "state profile used in the manuscript replication"},
        {"Parameter": "Sample end year", "Value": SAMPLE_END_YEAR, "Role": "common official TiVA-window endpoint used in the manuscript replication"},
        {"Parameter": "Validation mode", "Value": VALIDATION_MODE, "Role": "benchmark compares recomputed outputs with frozen validation targets"},
    ]
)


## Source inventory and coverage

The first audit step checks which frozen source files are available and what year range each file actually contains. This separates source availability from the modelling choice to use a common TiVA-linked state window.


In [2]:
source_rows = []
for path in sorted(FROZEN_SOURCES.glob("*.csv")):
    frame = pd.read_csv(path)
    years = pd.to_numeric(frame.get("year"), errors="coerce") if "year" in frame.columns else pd.Series(dtype=float)
    source_rows.append(
        {
            "Source file": path.name,
            "Rows": len(frame),
            "Countries": int(frame["country"].nunique()) if "country" in frame.columns else "",
            "First year": int(years.min()) if len(years.dropna()) else "",
            "Last year": int(years.max()) if len(years.dropna()) else "",
            "Columns": ", ".join(frame.columns[:8]),
        }
    )
source_inventory = pd.DataFrame(source_rows)
record("source files present", not source_inventory.empty, f"files={len(source_inventory)}")
display(source_inventory)


,Source file,Rows,Countries,First year,Last year,Columns
0,gdp_pc_current_pps.csv,810,27,1995,2024,"freq, unit, na_item, geo, time, gdp_pc_current..."
1,gdp_pc_real_index_2020.csv,805,27,1995,2024,"freq, unit, na_item, geo, time, gdp_pc_real_in..."
2,hh_total_financial_assets.csv,803,27,1995,2024,"freq, unit, co_nco, sector, finpos, na_item, g..."
3,hh_total_financial_liabilities.csv,803,27,1995,2024,"freq, unit, co_nco, sector, finpos, na_item, g..."
4,nominal_gdp.csv,810,27,1995,2024,"freq, unit, na_item, geo, time, nominal_gdp_mi..."
5,oecd_tiva_import_content_gfcf_cons_1995_2022.csv,1512,27,1995,2022,"country, year, measure, domestic_value_added_s..."
6,oecd_tiva_mainsh_domestic_va_shares_gfcf_cons_...,1512,,,,"STRUCTURE, STRUCTURE_ID, STRUCTURE_NAME, ACTIO..."


## Build state-variable source measures

The code below reconstructs the three manuscript replacement state variables from source files: investment import content, real PPP income and household balance-sheet fragility. It shows the economic transformations before the model-input panel is written.


In [3]:
def source_frame(name, value_col):
    frame = pd.read_csv(FROZEN_SOURCES / name)
    return frame[["country", "year", value_col]].copy()

state_source = source_frame("nominal_gdp.csv", "nominal_gdp_mio_eur")
for filename, value in [
    ("gdp_pc_current_pps.csv", "gdp_pc_current_pps"),
    ("gdp_pc_real_index_2020.csv", "gdp_pc_real_index_2020"),
    ("hh_total_financial_assets.csv", "hh_total_financial_assets_mio_eur"),
    ("hh_total_financial_liabilities.csv", "hh_total_financial_liabilities_mio_eur"),
]:
    state_source = state_source.merge(source_frame(filename, value), on=["country", "year"], how="outer", validate="one_to_one")

pps_2020 = state_source.loc[state_source["year"].eq(2020), ["country", "gdp_pc_current_pps"]].rename(
    columns={"gdp_pc_current_pps": "gdp_pc_2020_pps_anchor"}
)
state_source = state_source.merge(pps_2020, on="country", how="left", validate="many_to_one")
state_source["real_ppp_gdp_pc_2020pps"] = state_source["gdp_pc_2020_pps_anchor"] * state_source["gdp_pc_real_index_2020"] / 100.0
state_source["log_real_ppp_gdp_pc_raw"] = np.where(
    state_source["real_ppp_gdp_pc_2020pps"].gt(0), np.log(state_source["real_ppp_gdp_pc_2020pps"]), np.nan
)
state_source["hh_net_financial_worth_to_gdp"] = (
    state_source["hh_total_financial_assets_mio_eur"] - state_source["hh_total_financial_liabilities_mio_eur"]
) / state_source["nominal_gdp_mio_eur"]
state_source["household_balance_sheet_fragility_raw"] = -state_source["hh_net_financial_worth_to_gdp"]
state_source[GDP_RAW] = state_source["log_real_ppp_gdp_pc_raw"]

tiva = pd.read_csv(FROZEN_SOURCES / "oecd_tiva_import_content_gfcf_cons_1995_2022.csv")
gfcf = tiva[tiva["measure"].eq("GFCF_VA_SH")][["country", "year", "import_content_share"]].rename(
    columns={"import_content_share": "investment_import_content_raw"}
)
state_source = state_source.merge(gfcf, on=["country", "year"], how="left", validate="one_to_one")
state_source = state_source[state_source["country"].isin(EU27)].sort_values(["country", "year"]).reset_index(drop=True)

poland_profile_raw = state_source.loc[state_source["country"].eq("POL") & state_source["year"].eq(PROFILE_YEAR), [
    "country",
    "year",
    "investment_import_content_raw",
    "household_balance_sheet_fragility_raw",
    "log_real_ppp_gdp_pc_raw",
]]
poland_profile_public = poland_profile_raw.rename(
    columns={
        "country": "Country",
        "year": "Profile year",
        "investment_import_content_raw": "Investment import content",
        "household_balance_sheet_fragility_raw": "Household balance-sheet fragility",
        "log_real_ppp_gdp_pc_raw": "Real PPP income, log level",
    }
)
display(poland_profile_public)
record("Poland state profile source row", len(poland_profile_raw) == 1, f"rows={len(poland_profile_raw)}")


,Country,Profile year,Investment import content,Household balance-sheet fragility,"Real PPP income, log level"
657,POL,2022,0.41308,-0.657148,10.184149


## Standardise state variables and validate model-input rebuild

The manuscript replication standardises state variables on the EU27 panel through the common official TiVA state window. The next cell writes the runtime model input and compares it with the frozen validation input shipped in the package.


In [4]:
repro = import_module(CODE / "run_full_estimator_repro.py", "public_repro_steps")
repro.PROFILE_YEAR = PROFILE_YEAR
repro.SAMPLE_END_YEAR = SAMPLE_END_YEAR
repro.VALIDATION_MODE = VALIDATION_MODE

rebuilt_model_input = repro.build_runtime_model_inputs()
model_input_qa = pd.read_csv(QA / "full_estimator_model_input_rebuild_qa.csv")
transformations = pd.read_csv(RECOMPUTED / "model_inputs/variant_transformations.csv")
state_summary = pd.read_csv(RESULTS / "source_rebuilt_adopted_state_variable_summary_20260514.csv")

transformations_public = transformations.copy()
transformations_public["State variable"] = transformations_public["z_column"].map(public_column_label)
transformations_public = transformations_public[["State variable", "sample_start", "sample_end", "mean", "sd", "nobs"]].rename(
    columns={
        "sample_start": "Sample start",
        "sample_end": "Sample end",
        "mean": "EU27 mean",
        "sd": "EU27 standard deviation",
        "nobs": "Observations",
    }
)
display(transformations_public)
state_summary_public = state_summary[[
    "state_variable",
    "sample_start",
    "sample_end",
    "n",
    "mean",
    "sd_population",
    "poland_value_2022",
    "poland_z_2022",
]].copy().rename(
    columns={
        "state_variable": "State variable",
        "sample_start": "Sample start",
        "sample_end": "Sample end",
        "n": "Observations",
        "mean": "EU27 mean",
        "sd_population": "EU27 standard deviation",
        "poland_value_2022": "Poland source value",
        "poland_z_2022": "Poland standardised value",
    }
)
display(state_summary_public)
model_input_qa_public = model_input_qa.copy()
model_input_qa_public["check"] = model_input_qa_public["check"].map(public_validation_label)
display(model_input_qa_public)
record("source rebuilt model input", model_input_qa["status"].eq("PASS").all(), f"{len(model_input_qa)} source rebuild checks passed")


,State variable,Sample start,Sample end,EU27 mean,EU27 standard deviation,Observations
0,real PPP income standardised value,2004,2022,10.210231,0.383667,513
1,investment import content standardised value,2004,2022,0.429329,0.100846,513
2,household balance-sheet standardised value,2004,2022,-1.112256,0.591977,513


,State variable,Sample start,Sample end,Observations,EU27 mean,EU27 standard deviation,Poland source value,Poland standardised value
0,Investment import content,2004,2022,513,0.429329,0.100846,0.413080,-0.161130
1,Household net financial worth,2004,2022,513,-1.112256,0.591977,-0.657148,0.768794
2,Real PPP income level,2004,2022,513,10.210231,0.383667,10.184149,-0.067981


,check,status,detail
0,source rebuild validation: investment import c...,PASS,max_abs_diff=0.000e+00; same_missing=True
1,source rebuild validation: investment import c...,PASS,max_abs_diff=4.995e-10; same_missing=True
2,source rebuild validation: lagged investment i...,PASS,max_abs_diff=4.995e-10; same_missing=True
3,source rebuild validation: household balance-s...,PASS,max_abs_diff=4.996e-10; same_missing=True
4,source rebuild validation: household balance-s...,PASS,max_abs_diff=4.999e-10; same_missing=True
5,source rebuild validation: lagged household ba...,PASS,max_abs_diff=4.999e-10; same_missing=True
6,source rebuild validation: real PPP income sou...,PASS,max_abs_diff=4.990e-10; same_missing=True
7,source rebuild validation: real PPP income sta...,PASS,max_abs_diff=4.993e-10; same_missing=True
8,source rebuild validation: lagged real PPP income,PASS,max_abs_diff=4.993e-10; same_missing=True


## Build modelling panel and run the fifteen-candidate screen

This step estimates the full candidate screen from the runtime model input. The displayed table is produced from the fresh screen, not from the frozen benchmark.


In [5]:
with contextlib.redirect_stdout(StringIO()):
    feature_mod, screen = repro.run_feature_screen()
selected = (
    screen.loc[screen[GATE_STATUS_COL].eq(PASS_GATE), ["spec_id", "features"]]
    .copy()
    .sort_values("spec_id")
    .reset_index(drop=True)
)

feature_labels = {
    "trade": "investment import content",
    "debt": "public debt",
    "liq": "household balance-sheet fragility",
    GDP_KEY: "real PPP income",
}
def public_features(features):
    return " + ".join(feature_labels.get(part, part) for part in str(features).split("+") if part)

def public_regressor(term):
    if term == GI_SHOCK:
        return "public-investment shock"
    if term == "shock_" + "G_C":
        return "public-consumption shock control"
    if str(term).startswith(GI_INTERACTION_PREFIX):
        return "public-investment interaction: " + public_features(str(term).replace(GI_INTERACTION_PREFIX, "", 1))
    if str(term).endswith("_z_lag1"):
        return "lagged state/control: " + public_features(str(term).replace("_z_lag1", ""))
    return str(term).replace("_", " ")

screen_display = screen[["features", GATE_STATUS_COL, GATE_REASON_COL, "h8_design_rank", "h8_condition_number", "p_wald_y_h8", "max_abs_poland_z"]].copy()
screen_display["State variables"] = screen_display["features"].map(public_features)
screen_display["Screen result"] = screen_display[GATE_STATUS_COL].map(public_screen_status)
screen_display["Reason"] = screen_display[GATE_STATUS_COL].map(public_screen_reason)
screen_display = screen_display.drop(columns=["features", GATE_STATUS_COL, GATE_REASON_COL])
display(screen_display)
record("retained evaluations from fresh screen", len(selected) == 2, f"retained={selected['features'].map(public_features).tolist()}")


,h8_design_rank,h8_condition_number,p_wald_y_h8,max_abs_poland_z,State variables,Screen result,Reason
0,8,70.880814,0.003771,0.161130,investment import content,retained,passed all documented quality gates
1,8,68.369588,0.013043,0.768794,household balance-sheet fragility,retained,passed all documented quality gates
2,8,70.735072,0.119830,0.376724,public debt,not retained,did not pass at least one documented quality gate
3,8,68.794480,0.463297,0.067981,real PPP income,not retained,did not pass at least one documented quality gate
4,12,72.200433,0.658554,0.376724,investment import content + public debt + real...,not retained,did not pass at least one documented quality gate
5,10,71.101604,0.675856,0.161130,investment import content + real PPP income,not retained,did not pass at least one documented quality gate
6,12,71.201494,0.690439,0.768794,investment import content + household balance-...,not retained,did not pass at least one documented quality gate
7,10,70.963703,0.696225,0.768794,investment import content + household balance-...,not retained,did not pass at least one documented quality gate
8,10,70.793723,0.804693,0.768794,public debt + household balance-sheet fragility,not retained,did not pass at least one documented quality gate
9,10,72.113452,0.895313,0.376724,investment import content + public debt,not retained,did not pass at least one documented quality gate


## Inspect a design matrix before interpreting p-values

The screen p-values come from a horizon-specific local-projection design. This cell exposes the regression sample, columns and numerical rank for one retained evaluation at the diagnostic horizon.


In [6]:
grid_mod = feature_mod.load_grid_module()
base = grid_mod.load_base_module()
v3, work, feature_panel = feature_mod.load_work(grid_mod, base)

audit_spec = selected.iloc[0]
audit_features = tuple(str(audit_spec["features"]).split("+"))
feature_mod.configure(grid_mod, base, audit_features, "audit_design_matrix")
cols = base.x_columns(False)
horizon = 8
sample = work[work["year"].between(*v3.shock_window(horizon))].copy()
needed = [f"y_dyn_h{horizon}", *cols, "country", "year"]
used = sample.dropna(subset=needed).sort_values(["country", "year"]).reset_index(drop=True)
projector = v3.FEProjector(used["country"], used["year"])
x_res = projector.residualize(used[cols].to_numpy(dtype=float))
design_audit = pd.DataFrame(
    [
        {
            "Evaluation": public_features(audit_spec["features"]),
            "Horizon": horizon,
            "Observations": len(used),
            "Countries": used["country"].nunique(),
            "Years": f"{int(used['year'].min())}-{int(used['year'].max())}",
            "Regressor columns": ", ".join(public_regressor(col) for col in cols),
            "Residualised design rank": int(np.linalg.matrix_rank(x_res)),
            "Column count": len(cols),
        }
    ]
)
display(design_audit)
record("diagnostic design matrix full rank", int(np.linalg.matrix_rank(x_res)) == len(cols), design_audit.to_string(index=False))


,Evaluation,Horizon,Observations,Countries,Years,Regressor columns,Residualised design rank,Column count
0,household balance-sheet fragility,8,351,27,2004-2016,"public-investment shock, public-consumption sh...",8,8


## Fit one local-projection regression visibly

This cell performs the actual fixed-effect local-projection fit for the same retained evaluation and horizon. It exposes the residualised design, coefficient vector, covariance matrix, standard errors and pointwise p-values used by the screen rather than treating a generated table as evidence.


In [7]:
visible_fit = feature_mod.regression_fit(v3, sample, f"y_dyn_h{horizon}", cols)
record("visible local-projection regression fit", visible_fit.get("status") == "OK", f"status={visible_fit.get('status')}; nobs={visible_fit.get('nobs')}")

visible_beta = np.asarray(visible_fit["beta"], dtype=float)
visible_cov = np.asarray(visible_fit["cov"], dtype=float)
visible_terms = [GI_SHOCK, *[GI_INTERACTION_PREFIX + feature for feature in audit_features]]
visible_rows = []
for term in visible_terms:
    idx = cols.index(term)
    se = math.sqrt(max(float(visible_cov[idx, idx]), 0.0))
    coefficient = float(visible_beta[idx])
    z_stat = coefficient / se if se > 0 else math.nan
    pointwise_p = math.erfc(abs(z_stat) / math.sqrt(2.0)) if math.isfinite(z_stat) else math.nan
    visible_rows.append(
        {
            "Evaluation": public_features(audit_spec["features"]),
            "Horizon": horizon,
            "Term": public_regressor(term),
            "Coefficient": coefficient,
            "Standard error": se,
            "z statistic": z_stat,
            "Pointwise p-value": pointwise_p,
            "Observations": visible_fit["nobs"],
            "Design rank": visible_fit["rank"],
        }
    )
visible_regression_output = pd.DataFrame(visible_rows)
display(visible_regression_output)

interaction_idx = [cols.index(GI_INTERACTION_PREFIX + feature) for feature in audit_features]
interaction_beta = visible_beta[interaction_idx]
interaction_cov = visible_cov[np.ix_(interaction_idx, interaction_idx)]
visible_wald_stat = float(interaction_beta.T @ np.linalg.pinv(interaction_cov) @ interaction_beta)
visible_wald_p = float(feature_mod.chi2.sf(visible_wald_stat, len(interaction_idx)))
visible_wald = pd.DataFrame(
    [
        {
            "Evaluation": public_features(audit_spec["features"]),
            "Horizon": horizon,
            "Wald statistic": visible_wald_stat,
            "Degrees of freedom": len(interaction_idx),
            "Wald p-value": visible_wald_p,
            "Covariance rank": int(np.linalg.matrix_rank(interaction_cov)),
        }
    ]
)
display(visible_wald)
record("visible coefficient p-values finite", visible_regression_output["Pointwise p-value"].notna().all(), visible_regression_output.to_string(index=False))
record("visible interaction covariance full rank", int(np.linalg.matrix_rank(interaction_cov)) == len(interaction_idx), visible_wald.to_string(index=False))


,Evaluation,Horizon,Term,Coefficient,Standard error,z statistic,Pointwise p-value,Observations,Design rank
0,household balance-sheet fragility,8,public-investment shock,0.003761,0.007565,0.497231,0.619026,351,8
1,household balance-sheet fragility,8,public-investment interaction: household balan...,0.017351,0.006989,2.482581,0.013043,351,8


,Evaluation,Horizon,Wald statistic,Degrees of freedom,Wald p-value,Covariance rank
0,household balance-sheet fragility,8,6.163208,1,0.013043,1


## Recompute the eighth-horizon interaction tests

The next cell recomputes the diagnostic output-interaction Wald tests for all fifteen candidate subsets. These are the tests behind the retained-evaluation screen.


In [8]:
catalog = feature_mod.spec_catalog()
wald_rows = []
for row in catalog.itertuples(index=False):
    features = tuple(str(row.features).split("+"))
    feature_mod.configure(grid_mod, base, features, str(row.spec_id))
    result = feature_mod.output_interaction_wald(base, v3, work, features, horizon=8)
    wald_rows.append(
        {
            "State variables": public_features(row.features),
            "Wald statistic h8": result["wald_y_h8"],
            "p-value h8": result["p_wald_y_h8"],
            "Status": result["status"],
            "Observations": result["nobs"],
        }
    )
wald_audit = pd.DataFrame(wald_rows).sort_values("p-value h8").reset_index(drop=True)
display(wald_audit)
record("fifteen candidate Wald tests", len(wald_audit) == 15 and wald_audit["Status"].eq("OK").all(), f"rows={len(wald_audit)}")


,State variables,Wald statistic h8,p-value h8,Status,Observations
0,investment import content,8.390819,0.003771,OK,351
1,household balance-sheet fragility,6.163208,0.013043,OK,351
2,public debt,2.419546,0.119830,OK,351
3,real PPP income,0.537916,0.463297,OK,351
4,investment import content + public debt + real...,1.603688,0.658554,OK,351
5,investment import content + real PPP income,0.783549,0.675856,OK,351
6,investment import content + household balance-...,1.464710,0.690439,OK,351
7,investment import content + household balance-...,0.724165,0.696225,OK,351
8,public debt + household balance-sheet fragility,0.434589,0.804693,OK,351
9,investment import content + public debt,0.221163,0.895313,OK,351


## Estimate retained output and spending response paths

The retained evaluations are estimated after the screen. The displayed values are cumulative responses built from horizon-by-horizon incremental local-projection estimates.


In [9]:
polish_mod = import_module(CODE / "estimation/polish_output_spending_model.py", "public_polish_output_spending_visible")
polish_mod.DATA_DIR = WORK_DATA
polish_mod.RESULTS_DIR = RECOMPUTED / "polish_output_spending"
polish_mod.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
polish_mod.REFERENCES_DIR = REFERENCES
polish_mod.GRID_CODE = REFERENCES / "run_c_pl_feature_grid_base.py"
polish_mod.ROOT = ROOT
polish_mod.SPEC_VERSION = "mozdzen_" + "tiva2022_gfcf_realppp_networth_polish_output_20260513"
repro.patch_grid_profile(polish_mod)

polish_grid = polish_mod.load_grid_module()
polish_base, polish_v3, polish_work = polish_mod.load_work(polish_grid)
polish_path_blocks = []
for row in selected.itertuples(index=False):
    block = polish_mod.estimate_kernels(polish_grid, polish_base, polish_v3, polish_work, str(row.spec_id), str(row.features))
    block["profile_year_used"] = PROFILE_YEAR
    polish_path_blocks.append(block)
polish_paths = pd.concat(polish_path_blocks, ignore_index=True).sort_values(["spec_id", "horizon"]).reset_index(drop=True)

polish_paths["K_Y_from_visible_incremental_sum"] = polish_paths.groupby("features")["mu_Y_incremental"].cumsum()
polish_paths["K_G_from_visible_incremental_sum"] = polish_paths.groupby("features")["mu_G_incremental"].cumsum()
polish_paths["K_Y_visible_difference"] = polish_paths["K_Y_from_visible_incremental_sum"] - polish_paths["K_Y_cumulative"]
polish_paths["K_G_visible_difference"] = polish_paths["K_G_from_visible_incremental_sum"] - polish_paths["K_G_cumulative"]
canonical_polish_path_columns = [
    "spec_version",
    "spec_id",
    "features",
    "profile_label",
    "horizon",
    "mu_Y_incremental",
    "se_Y_incremental",
    "mu_G_incremental",
    "se_G_incremental",
    "beta_scale_action",
    "se_beta_scale_action",
    "action_denom_t",
    "nobs",
    "country_n",
    "year_min_effective",
    "year_max_effective",
    "rank_X",
    "status_Y",
    "status_G",
    "z_trade",
    "K_Y_cumulative",
    "K_G_cumulative",
    "cumulative_output_to_spending_ratio",
    "ci95_low_K_Y_cumulative_naive",
    "ci95_high_K_Y_cumulative_naive",
    "z_liq",
    "profile_year_used",
]
polish_paths_canonical = polish_paths[canonical_polish_path_columns].copy()
repro.write_frame(polish_paths_canonical, RECOMPUTED / "polish_output_spending/polish_output_spending_paths.csv")
repro.write_frame(polish_mod.make_summary(polish_paths_canonical), RECOMPUTED / "polish_output_spending/polish_output_spending_h8_summary.csv")
polish_qa = pd.DataFrame(
    [
        {
            "check": "all horizons present for retained evaluations",
            "status": "PASS" if polish_paths.groupby("spec_id")["horizon"].nunique().min() == 9 else "FAIL",
            "detail": "h0-h8 per retained evaluation",
        },
        {
            "check": "all output and spending regressions returned OK",
            "status": "PASS" if polish_paths["status_Y"].eq("OK").all() and polish_paths["status_G"].eq("OK").all() else "FAIL",
            "detail": "status_Y/status_G over all retained horizons",
        },
        {
            "check": "cumulative K paths equal visible sums of incremental estimates",
            "status": "PASS" if polish_paths[["K_Y_visible_difference", "K_G_visible_difference"]].abs().max().max() <= 1e-10 else "FAIL",
            "detail": "K_Y and K_G reconstructed inside this notebook cell",
        },
    ]
)
repro.write_frame(polish_qa, RECOMPUTED / "polish_output_spending/qa_checks.csv")

polish_visible_paths = polish_paths[[
    "features",
    "horizon",
    "mu_Y_incremental",
    "mu_G_incremental",
    "K_Y_cumulative",
    "K_G_cumulative",
    "cumulative_output_to_spending_ratio",
    "nobs",
    "country_n",
    "year_min_effective",
    "year_max_effective",
    "status_Y",
    "status_G",
]].copy()
polish_visible_paths["Evaluation"] = polish_visible_paths["features"].map(public_features)
polish_visible_paths = polish_visible_paths.drop(columns=["features"])
display(polish_visible_paths)
display(polish_qa)

polish_h8 = polish_paths.loc[polish_paths["horizon"].eq(8), [
    "features",
    "K_Y_cumulative",
    "K_G_cumulative",
    "cumulative_output_to_spending_ratio",
    "nobs",
    "country_n",
    "year_min_effective",
    "year_max_effective",
]].copy()
polish_h8["Evaluation"] = polish_h8["features"].map(public_features)
polish_h8 = polish_h8.drop(columns=["features"])
display(polish_h8)
record("retained response paths h0-h8", polish_paths.groupby("features")["horizon"].nunique().min() == 9, "h0-h8 per retained evaluation")
record("visible cumulative K arithmetic", polish_qa["status"].eq("PASS").all(), polish_qa.to_string(index=False))


,horizon,mu_Y_incremental,mu_G_incremental,K_Y_cumulative,K_G_cumulative,cumulative_output_to_spending_ratio,nobs,country_n,year_min_effective,year_max_effective,status_Y,status_G,Evaluation
0,0,1.194279,1.000000,1.194279,1.000000,1.194279,531,27,2004,2023,OK,OK,household balance-sheet fragility
1,1,1.171577,0.054162,2.365856,1.054162,2.244301,531,27,2004,2023,OK,OK,household balance-sheet fragility
2,2,0.096277,-0.213520,2.462134,0.840642,2.928875,506,27,2004,2022,OK,OK,household balance-sheet fragility
3,3,-0.533015,-0.138219,1.929118,0.702422,2.746380,481,27,2004,2021,OK,OK,household balance-sheet fragility
4,4,-0.362489,-0.026826,1.566629,0.675596,2.318886,456,27,2004,2020,OK,OK,household balance-sheet fragility
5,5,0.047081,0.002667,1.613711,0.678263,2.379182,431,27,2004,2019,OK,OK,household balance-sheet fragility
6,6,0.015449,0.000103,1.629160,0.678366,2.401594,405,27,2004,2018,OK,OK,household balance-sheet fragility
7,7,0.144639,0.021297,1.773798,0.699663,2.535217,378,27,2004,2017,OK,OK,household balance-sheet fragility
8,8,0.384145,0.046344,2.157944,0.746008,2.892656,351,27,2004,2016,OK,OK,household balance-sheet fragility
9,0,1.387360,1.000000,1.387360,1.000000,1.387360,531,27,2004,2023,OK,OK,investment import content


,check,status,detail
0,all horizons present for retained evaluations,PASS,h0-h8 per retained evaluation
1,all output and spending regressions returned OK,PASS,status_Y/status_G over all retained horizons
2,cumulative K paths equal visible sums of incre...,PASS,K_Y and K_G reconstructed inside this notebook...


,K_Y_cumulative,K_G_cumulative,cumulative_output_to_spending_ratio,nobs,country_n,year_min_effective,year_max_effective,Evaluation
8,2.157944,0.746008,2.892656,351,27,2004,2016,household balance-sheet fragility
17,1.838266,0.694029,2.648686,351,27,2004,2016,investment import content


## Estimate direct debt response and institutional debt-accounting paths

The debt block uses the retained output and spending paths. It estimates the direct debt-to-GDP response and applies the institutional debt recursion to the three-year investment action.


In [10]:
debt_mod_audit = import_module(CODE / "estimation/debt_accounting_model.py", "public_debt_accounting_model_audit_cell")
debt_mod_audit.DATA_DIR = WORK_DATA
debt_mod_audit.RESULTS_DIR = RECOMPUTED / "debt_accounting_audit_cell"
debt_mod_audit.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
debt_mod_audit.REFERENCES_DIR = REFERENCES
debt_mod_audit.V3_CODE = REFERENCES / "project_context/ciaffi_canonical_estimation_v3_short_rate.py"
debt_mod_audit.ROOT = ROOT
debt_mod_audit.SPEC_VERSION = "public_notebook_visible_direct_debt_fit"
debt_mod_audit.PANEL_END_YEAR = PROFILE_YEAR

debt_v3_audit = debt_mod_audit.load_v3_module()
debt_work_audit = debt_mod_audit.prepare_work(debt_v3_audit)
audit_features_text = str(audit_spec["features"])
audit_debt_features = debt_mod_audit.parse_features(audit_features_text)
audit_z_values = debt_mod_audit.feature_values(audit_features_text)
visible_direct_debt_fit = debt_mod_audit.fit_direct_dy_ratio(
    debt_v3_audit,
    debt_work_audit,
    audit_debt_features,
    audit_z_values,
    horizon,
)
visible_direct_debt = pd.DataFrame([{**{"Evaluation": public_features(audit_features_text), "Horizon": horizon}, **visible_direct_debt_fit}])
display(visible_direct_debt)
record("visible direct debt LP fit", visible_direct_debt_fit["status_DY_initial"] == "OK", visible_direct_debt.to_string(index=False))


,Evaluation,Horizon,direct_DY_initial_action,se_direct_DY_initial_action,status_DY_initial,nobs,country_n,year_min_effective,year_max_effective,rank_X
0,household balance-sheet fragility,8,-1.869372,0.918368,OK,351,27,2004,2016,8


In [11]:
debt_mod_visible = import_module(CODE / "estimation/debt_accounting_model.py", "public_debt_accounting_model_visible")
debt_mod_visible.DATA_DIR = WORK_DATA
debt_mod_visible.RESULTS_DIR = RECOMPUTED / "debt_accounting"
debt_mod_visible.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
debt_mod_visible.REFERENCES_DIR = REFERENCES
debt_mod_visible.V3_CODE = REFERENCES / "project_context/ciaffi_canonical_estimation_v3_short_rate.py"
debt_mod_visible.ROOT = ROOT
debt_mod_visible.SPEC_VERSION = "mozdzen_" + "tiva2022_gfcf_realppp_networth_debt_20260513"
debt_mod_visible.PANEL_END_YEAR = PROFILE_YEAR

debt_v3_visible = debt_mod_visible.load_v3_module()
debt_work_visible = debt_mod_visible.prepare_work(debt_v3_visible)
direct_debt_paths = debt_mod_visible.estimate_direct_debt_kernels(debt_v3_visible, debt_work_visible, selected)
program_paths = debt_mod_visible.build_program_paths(polish_paths, direct_debt_paths, selected)
dsm_inputs = debt_mod_visible.load_dsm_inputs()
baseline_reproduction = debt_mod_visible.reproduce_baseline(dsm_inputs)
dsa_paths = debt_mod_visible.simulate_dsa(program_paths, dsm_inputs)
debt_summary_internal = debt_mod_visible.make_2036_summary(program_paths, dsa_paths)

repro.write_frame(direct_debt_paths, RECOMPUTED / "debt_accounting/direct_dy_initial_action_paths.csv")
repro.write_frame(program_paths, RECOMPUTED / "debt_accounting/three_year_program_paths.csv")
repro.write_frame(baseline_reproduction, RECOMPUTED / "debt_accounting/baseline_reproduction.csv")
repro.write_frame(dsa_paths, RECOMPUTED / "debt_accounting/dsa_debt_paths.csv")
repro.write_frame(debt_summary_internal, RECOMPUTED / "debt_accounting/polish_debt_2036_summary.csv")
debt_qa = pd.DataFrame(
    [
        {"check": "direct debt kernels present for retained evaluations", "status": "PASS" if not direct_debt_paths.empty else "FAIL", "detail": f"rows={len(direct_debt_paths)}"},
        {"check": "program paths present", "status": "PASS" if not program_paths.empty else "FAIL", "detail": f"rows={len(program_paths)}"},
        {
            "check": "baseline reproduced within tolerance",
            "status": "PASS" if baseline_reproduction["abs_diff_pp"].max() <= debt_mod_visible.DEBT_BASELINE_REPRO_TOL_PP else "FAIL",
            "detail": f"max_abs_diff={baseline_reproduction['abs_diff_pp'].max():.3e}",
        },
    ]
)
repro.write_frame(debt_qa, RECOMPUTED / "debt_accounting/qa_checks.csv")

direct_debt_display = direct_debt_paths[direct_debt_paths["horizon"].isin([0, 1, 2, 5, 8])].copy()
direct_debt_display["Evaluation"] = direct_debt_display["features"].map(public_features)
display(direct_debt_display[[
    "Evaluation",
    "horizon",
    "direct_DY_initial_action",
    "se_direct_DY_initial_action",
    "nobs",
    "country_n",
    "year_min_effective",
    "year_max_effective",
    "status_DY_initial",
]])
program_display = program_paths[program_paths["horizon_from_2028"].isin([0, 1, 2, 5, 8])].copy()
program_display["Evaluation"] = program_display["features"].map(public_features)
program_display["Scenario"] = program_display["scenario_sign"].map({"cut": "public-investment cuts", "expansion": "public-investment expansion"})
display(program_display[[
    "Evaluation",
    "Scenario",
    "year",
    "horizon_from_2028",
    "fiscal_action_cut_units_pp",
    "Y_shortfall_pct",
    "direct_discretionary_PB_level_pp",
    "direct_DY_LP_margin_initial_action_pp",
]].head(20))
display(debt_qa)

debt_summary_display = debt_summary_internal[[
    "features",
    "scenario",
    "dsa_margin_vs_baseline_pp",
    "direct_DY_LP_margin_pp",
]].copy()
debt_summary_display["Evaluation"] = debt_summary_display["features"].map(public_features)
debt_summary_display = debt_summary_display.drop(columns=["features"])
display(debt_summary_display)
record("Polish debt endpoint summary", not debt_summary_internal.empty, f"rows={len(debt_summary_internal)}")
record("visible debt-accounting objects", debt_qa["status"].eq("PASS").all(), debt_qa.to_string(index=False))


,Evaluation,horizon,direct_DY_initial_action,se_direct_DY_initial_action,nobs,country_n,year_min_effective,year_max_effective,status_DY_initial
0,household balance-sheet fragility,0,-1.403955,0.668558,531,27,2004,2023,OK
1,household balance-sheet fragility,1,-2.441563,0.931475,531,27,2004,2023,OK
2,household balance-sheet fragility,2,-2.271027,1.140025,506,27,2004,2022,OK
5,household balance-sheet fragility,5,-0.021217,0.822363,431,27,2004,2019,OK
8,household balance-sheet fragility,8,-1.869372,0.918368,351,27,2004,2016,OK
9,investment import content,0,-1.579393,0.593777,531,27,2004,2023,OK
10,investment import content,1,-2.512161,0.869154,531,27,2004,2023,OK
11,investment import content,2,-2.252854,1.234821,506,27,2004,2022,OK
14,investment import content,5,1.330592,1.166787,431,27,2004,2019,OK
17,investment import content,8,-0.807901,0.988298,351,27,2004,2016,OK


,Evaluation,Scenario,year,horizon_from_2028,fiscal_action_cut_units_pp,Y_shortfall_pct,direct_discretionary_PB_level_pp,direct_DY_LP_margin_initial_action_pp
0,household balance-sheet fragility,public-investment cuts,2028,0,1.0,1.194279,1.000000,1.403955
1,household balance-sheet fragility,public-investment cuts,2029,1,1.0,3.560136,2.054162,3.845518
2,household balance-sheet fragility,public-investment cuts,2030,2,1.0,6.022269,2.894803,6.116545
5,household balance-sheet fragility,public-investment cuts,2033,5,0.0,5.109459,2.056281,2.188124
8,household balance-sheet fragility,public-investment cuts,2036,8,0.0,5.560902,2.124037,4.309320
9,household balance-sheet fragility,public-investment expansion,2028,0,-1.0,-1.194279,-1.000000,-1.403955
10,household balance-sheet fragility,public-investment expansion,2029,1,-1.0,-3.560136,-2.054162,-3.845518
11,household balance-sheet fragility,public-investment expansion,2030,2,-1.0,-6.022269,-2.894803,-6.116545
14,household balance-sheet fragility,public-investment expansion,2033,5,-0.0,-5.109459,-2.056281,-2.188124
17,household balance-sheet fragility,public-investment expansion,2036,8,-0.0,-5.560902,-2.124037,-4.309320


,check,status,detail
0,direct debt kernels present for retained evalu...,PASS,rows=18
1,program paths present,PASS,rows=36
2,baseline reproduced within tolerance,PASS,max_abs_diff=1.472e-02


,scenario,dsa_margin_vs_baseline_pp,direct_DY_LP_margin_pp,Evaluation
0,three_1pp_cut_2028_2030,6.503118,4.309320,household balance-sheet fragility
1,three_1pp_expansion_2028_2030,-6.097918,-4.309320,household balance-sheet fragility
2,three_1pp_cut_2028_2030,4.890651,1.729582,investment import content
3,three_1pp_expansion_2028_2030,-4.636257,-1.729582,investment import content


## Produce coefficient-level estimation output

This cell writes the coefficient tables used by Appendix D and then computes the p-value disclosure from those coefficient files. The count is generated here, not typed as a target.


In [12]:
with contextlib.redirect_stdout(StringIO()):
    repro.make_estimation_output_tables(feature_mod, screen, polish_paths)
eu27_coef = pd.read_csv(RECOMPUTED / "estimation_output/eu27_benchmark_regression_coefficients.csv")
retained_coef = pd.read_csv(RECOMPUTED / "estimation_output/retained_regression_coefficients.csv")

eu27_displayed = eu27_coef[eu27_coef["term"].eq(GI_SHOCK)].copy()
retained_displayed = retained_coef[
    retained_coef["term"].eq(GI_SHOCK) | retained_coef["term"].str.startswith(GI_INTERACTION_PREFIX, na=False)
].copy()
displayed_pvalues = pd.concat([eu27_displayed, retained_displayed], ignore_index=True)
pvalue_disclosure = pd.DataFrame(
    [
        {
            "Displayed pointwise p-values": len(displayed_pvalues),
            "p-values above 0.1": int((displayed_pvalues["p_value"] > 0.1).sum()),
            "Minimum p-value": displayed_pvalues["p_value"].min(),
            "Maximum p-value": displayed_pvalues["p_value"].max(),
        }
    ]
)
display(pvalue_disclosure)
displayed_pvalues_public = displayed_pvalues[["path", "features", "outcome", "horizon", "term", "coefficient", "std_error", "p_value", "nobs", "country_n", "year_min", "year_max"]].copy()
displayed_pvalues_public["State variables"] = displayed_pvalues_public["features"].map(public_features)
displayed_pvalues_public["Term"] = displayed_pvalues_public["term"].map(public_regressor)
displayed_pvalues_public = displayed_pvalues_public.drop(columns=["features", "term"])
display(displayed_pvalues_public.head(18))
record("Appendix D p-value disclosure computed", len(displayed_pvalues) == 90, f"rows={len(displayed_pvalues)}")


,Displayed pointwise p-values,p-values above 0.1,Minimum p-value,Maximum p-value
0,90,59,0.0,0.971718


,path,outcome,horizon,coefficient,std_error,p_value,nobs,country_n,year_min,year_max,State variables,Term
0,EU27 panel benchmark,output,0,0.046864,0.010779,0.000014,557,27,2004,2024,linear,public-investment shock
1,EU27 panel benchmark,output,1,0.037713,0.012700,0.002981,531,27,2004,2023,linear,public-investment shock
2,EU27 panel benchmark,output,2,0.005842,0.010008,0.559393,506,27,2004,2022,linear,public-investment shock
3,EU27 panel benchmark,output,3,-0.010554,0.011607,0.363193,481,27,2004,2021,linear,public-investment shock
4,EU27 panel benchmark,output,4,-0.008788,0.013563,0.517042,456,27,2004,2020,linear,public-investment shock
5,EU27 panel benchmark,output,5,-0.000753,0.003890,0.846440,431,27,2004,2019,linear,public-investment shock
6,EU27 panel benchmark,output,6,0.002773,0.003987,0.486771,405,27,2004,2018,linear,public-investment shock
7,EU27 panel benchmark,output,7,0.004328,0.006953,0.533668,378,27,2004,2017,linear,public-investment shock
8,EU27 panel benchmark,output,8,0.009783,0.006059,0.106371,351,27,2004,2016,linear,public-investment shock
9,EU27 panel benchmark,spending,0,0.041126,0.001146,0.000000,557,27,2004,2024,linear,public-investment shock


## Rebuild manuscript-facing tables and figures

After the estimator and accounting steps finish, the notebook rebuilds the public tables and figures from `results/recomputed/`.


In [13]:
with contextlib.redirect_stdout(StringIO()):
    runpy.run_path(str(CODE / "build_public_tables_figures.py"), run_name="__main__")
table_qa = pd.read_csv(QA / "public_tables_figures_qa_20260514.csv")
display(table_qa)
record("public tables and figures", table_qa["status"].eq("PASS").all(), table_qa.to_string(index=False))


,check,status,detail
0,source_code_rows,PASS,13
1,state_rows,PASS,4
2,screen_rows,PASS,15
3,retained_specs,PASS,investment import content; household net finan...
4,debt_rows,PASS,4
5,debt_table_includes_eu27_benchmark,PASS,"EU27 panel benchmark,Polish evaluation based o..."
6,debt_decomposition_rows,PASS,54
7,eu27_annual_debt_decomposition_rows,PASS,18
8,eu27_annual_debt_decomposition_actions,PASS,"Cut,Expansion"
9,figure2_includes_eu27_benchmark,PASS,EU27 benchmark path recomputed by code/run_ful...


## Bridge from regression output to K paths

The bridge table is now read after the estimation and table-generation steps above. It is a presentation of objects computed in this notebook run.


In [14]:
bridge = pd.read_csv(TABLES / "estimation_response_bridge_by_horizon.csv")
bridge_paths = pd.read_csv(TABLES / "estimation_response_bridge_paths.csv")
bridge_sample = pd.read_csv(TABLES / "estimation_response_bridge_sample.csv")
h8_bridge = bridge[bridge["Horizon"].eq(8)].copy()
display(h8_bridge)
display(bridge_paths)
record("bridge h8 rows", len(h8_bridge) == 3, f"rows={len(h8_bridge)}")


,Path,Horizon,Incremental output response,Cumulative K_Y,Incremental spending response,Cumulative K_G,K_Y/K_G,Observations,Countries,Years
8,EU27 panel benchmark,8,+0.229 (0.138),2.108,+0.081 (0.040),0.757,2.786,351,27,2004-2016
17,Household net financial worth,8,+0.384 (0.071),2.158,+0.046 (0.036),0.746,2.893,351,27,2004-2016
26,Investment import content,8,+0.343 (0.091),1.838,+0.075 (0.038),0.694,2.649,351,27,2004-2016


,Path,Horizon,Incremental output response,Cumulative K_Y,Incremental spending response,Cumulative K_G,K_Y/K_G
0,EU27 panel benchmark,0,+1.140 (0.256),1.140,+1.000 (0.000),1.000,1.140
1,EU27 panel benchmark,1,+0.916 (0.315),2.056,+0.004 (0.088),1.004,2.049
2,EU27 panel benchmark,2,+0.142 (0.244),2.198,-0.174 (0.049),0.829,2.650
3,EU27 panel benchmark,3,-0.257 (0.277),1.941,-0.131 (0.062),0.698,2.780
4,EU27 panel benchmark,4,-0.213 (0.324),1.728,-0.041 (0.060),0.657,2.629
5,EU27 panel benchmark,5,-0.018 (0.094),1.710,-0.003 (0.024),0.654,2.614
6,EU27 panel benchmark,6,+0.066 (0.096),1.776,-0.007 (0.036),0.648,2.743
7,EU27 panel benchmark,7,+0.102 (0.163),1.879,+0.028 (0.051),0.676,2.779
8,EU27 panel benchmark,8,+0.229 (0.138),2.108,+0.081 (0.040),0.757,2.786
9,Household net financial worth,0,+1.194 (0.337),1.194,+1.000 (0.000),1.000,1.194


## Debt endpoints and equal-weight arithmetic

The next table is the manuscript-facing 2036 endpoint table rebuilt from the recomputed debt outputs. The equal-weight row is the arithmetic average of the two retained Polish evaluations.


In [15]:
debt = pd.read_csv(TABLES / "debt_2036.csv")
decomp = pd.read_csv(TABLES / "annual_debt_decomposition.csv")
display(debt)
display(decomp.head(18))

polish_only = debt[debt["Empirical path"].isin([
    "Polish evaluation based on investment import content",
    "Polish evaluation based on household net financial worth",
])].copy()
equal_row = debt[debt["Empirical path"].eq("Equal weight average across the two Polish evaluations")].iloc[0]
numeric_cols = [
    "Expansion, institutional debt equation",
    "Expansion, direct debt-to-GDP local-projection path",
    "Cut, institutional debt equation",
    "Cut, direct debt-to-GDP local-projection path",
]
computed_equal = polish_only[numeric_cols].astype(float).mean()
equal_check = pd.DataFrame({"Column": numeric_cols, "Notebook arithmetic": computed_equal.values, "Displayed equal weight": equal_row[numeric_cols].astype(float).values})
equal_check["Difference"] = equal_check["Notebook arithmetic"] - equal_check["Displayed equal weight"]
display(equal_check)
record("equal-weight arithmetic", equal_check["Difference"].abs().max() <= 0.051, equal_check.to_string(index=False))


,Empirical path,"Expansion, institutional debt equation","Expansion, direct debt-to-GDP local-projection path","Cut, institutional debt equation","Cut, direct debt-to-GDP local-projection path"
0,EU27 panel benchmark,-6.5,-3.7,7.0,3.7
1,Polish evaluation based on investment import c...,-4.6,-1.7,4.9,1.7
2,Polish evaluation based on household net finan...,-6.1,-4.3,6.5,4.3
3,Equal weight average across the two Polish eva...,-5.4,-3.0,5.7,3.0


,Empirical path,Action,Year,Baseline debt ratio,Scenario debt ratio,Debt margin,"Output effect, GDP level",Direct primary-balance effect,Cyclical primary-balance feedback,Baseline primary balance,Scenario primary balance,Scenario nominal GDP growth,Snowball term,Stock-flow adjustment,Institutional debt margin,Direct debt-to-GDP LP margin
0,Investment import content,Expansion,2028,73.19,72.58,-0.61,-1.39,-1.00,0.67,-3.88,-4.21,6.89,-1.31,0.0,-0.61,-1.58
1,Investment import content,Expansion,2029,77.05,75.06,-1.98,-3.69,-2.01,1.77,-3.95,-4.19,7.63,-1.71,0.0,-1.98,-4.09
2,Investment import content,Expansion,2030,81.12,77.48,-3.64,-5.99,-2.85,2.88,-3.99,-3.96,7.50,-1.55,0.0,-3.64,-6.34
3,Investment import content,Expansion,2031,85.14,80.59,-4.55,-6.45,-2.53,3.10,-3.91,-3.34,5.78,-0.23,0.0,-4.55,-6.23
4,Investment import content,Expansion,2032,89.34,84.89,-4.44,-5.58,-2.13,2.68,-4.02,-3.47,4.55,0.83,0.0,-4.44,-3.24
5,Investment import content,Expansion,2033,93.42,89.58,-3.84,-4.50,-1.89,2.16,-3.84,-3.58,4.42,1.11,0.0,-3.84,0.35
6,Investment import content,Expansion,2034,97.62,94.13,-3.49,-3.95,-1.79,1.90,-3.86,-3.76,4.99,0.80,0.0,-3.49,1.26
7,Investment import content,Expansion,2035,101.99,98.30,-3.69,-4.01,-1.80,1.93,-3.88,-3.75,5.59,0.42,0.0,-3.69,0.41
8,Investment import content,Expansion,2036,106.79,102.16,-4.64,-4.63,-1.89,2.22,-3.89,-3.56,5.84,0.29,0.0,-4.64,-1.73
9,Investment import content,Cut,2028,73.19,73.84,0.64,1.39,1.00,-0.67,-3.88,-3.55,3.96,0.62,0.0,0.64,1.58


,Column,Notebook arithmetic,Displayed equal weight,Difference
0,"Expansion, institutional debt equation",-5.35,-5.4,0.05
1,"Expansion, direct debt-to-GDP local-projection...",-3.00,-3.0,0.00
2,"Cut, institutional debt equation",5.70,5.7,0.00
3,"Cut, direct debt-to-GDP local-projection path",3.00,3.0,0.00


## Validate recomputed outputs against frozen benchmark files

Validation happens at the end. It checks the fresh computation against frozen benchmark files with tolerance rules and does not replace the computation above.


In [16]:
validation = repro.validate_against_frozen()
validation_public = validation.copy()
validation_public["check"] = validation_public["check"].map(public_validation_label)
display(validation_public)
record("frozen benchmark validation", validation["status"].eq("PASS").all(), f"{len(validation)} benchmark comparisons passed")
repro.write_manifest()

selected_horizons = bridge[bridge["Horizon"].isin([0, 1, 2, 5, 8])].copy()
selected_horizons.to_csv(RESULTS / "notebook_selected_horizons.csv", index=False)
debt.to_csv(RESULTS / "notebook_debt_margins_2036.csv", index=False)

all_checks = pd.DataFrame(checks)
all_checks.to_csv(RESULTS / "notebook_check_summary.csv", index=False)
all_checks_public = all_checks.copy()
all_checks_public["detail"] = all_checks_public["detail"].map(public_validation_label)
display(all_checks_public)
if not all_checks["passed"].all():
    raise RuntimeError("Notebook reproduction checks failed")


,check,status,detail
0,candidate screen: feature_robustness_summary,PASS,max_numeric_diff=3.000e-09; bad_cols=
1,candidate screen: output_interaction_wald_h8,PASS,max_numeric_diff=3.000e-09; bad_cols=
2,candidate screen: output_interaction_multiplic...,PASS,max_numeric_diff=3.000e-09; bad_cols=
3,candidate screen: kernel_paths_all_horizons,PASS,max_numeric_diff=8.900e-08; bad_cols=
4,candidate screen: kernel_h8,PASS,max_numeric_diff=8.900e-08; bad_cols=
5,candidate screen: bootstrap_kernel_summary,PASS,max_numeric_diff=3.000e-09; bad_cols=
6,candidate screen: loo_kernel_summary,PASS,max_numeric_diff=1.000e-09; bad_cols=
7,candidate screen: time_block_kernel_summary,PASS,max_numeric_diff=3.000e-09; bad_cols=
8,Polish response path: paths,PASS,max_numeric_diff=2.400e-08; bad_cols=
9,Polish response path: h8_summary,PASS,max_numeric_diff=2.000e-09; bad_cols=


,check,returncode,passed,detail
0,source files present,0,True,files=7
1,Poland state profile source row,0,True,rows=1
2,source rebuilt model input,0,True,9 source rebuild checks passed
3,retained evaluations from fresh screen,0,True,"retained=['household balance-sheet fragility',..."
4,diagnostic design matrix full rank,0,True,Evaluation Horizon Ob...
5,visible local-projection regression fit,0,True,status=OK; nobs=351
6,visible coefficient p-values finite,0,True,Evaluation Horizon ...
7,visible interaction covariance full rank,0,True,Evaluation Horizon Wa...
8,fifteen candidate Wald tests,0,True,rows=15
9,retained response paths h0-h8,0,True,h0-h8 per retained evaluation
